# 08 — Training and Baselines
Train the GAT model with scenario-aware split and compare against persistence + train-mean baselines.


In [1]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
df = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "synthetic_snapshots.csv")
mapping = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "road_segment_mapping.csv")
edge_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "segment_graph_edges.csv")


In [2]:
from gat_traffic.validation import scenario_split

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

scenarios = sorted(df.scenario_id.unique())
train_s, val_s, test_s = scenario_split(scenarios, train_ratio=0.7, val_ratio=0.15, seed=SEED)

split_col = np.where(df.scenario_id.isin(train_s), "train", np.where(df.scenario_id.isin(val_s), "val", "test"))
df = df.assign(split=split_col)
print(df.groupby("split").agg(scenarios=("scenario_id","nunique"), snapshots=("snapshot_id","nunique")))


       scenarios  snapshots
split                      
test           9        300
train         42       1401
val            9        299


In [3]:
feat_cols = ["current_speed_kmh", "current_density", "queue_length", "capacity_factor"]
target_col = "target_travel_time_sec"

train_mask = df.split == "train"
feat_mean = df.loc[train_mask, feat_cols].mean().to_numpy()
feat_std = df.loc[train_mask, feat_cols].std().replace(0,1).to_numpy()
tgt_mean = float(df.loc[train_mask, target_col].mean())
tgt_std = float(df.loc[train_mask, target_col].std()) or 1.0

def norm_x(x):
    return (x - feat_mean) / feat_std

def norm_y(y):
    return (y - tgt_mean) / tgt_std


In [4]:
# Build compact tensors by snapshot for train/val/test
N = len(mapping)
edge_index = torch.tensor(edge_df[["src","dst"]].to_numpy().T, dtype=torch.long)

snapshot_ids = sorted(df.snapshot_id.unique())
by_snap = {sid: sdf.sort_values("road_segment_id") for sid, sdf in df.groupby("snapshot_id")}

X, Y, SPLIT = [], [], []
for sid in snapshot_ids:
    sdf = by_snap[sid]
    X.append(norm_x(sdf[feat_cols].to_numpy(dtype=np.float32)))
    Y.append(norm_y(sdf[target_col].to_numpy(dtype=np.float32)))
    SPLIT.append(sdf.split.iloc[0])

X = torch.tensor(np.stack(X), dtype=torch.float32)
Y = torch.tensor(np.stack(Y), dtype=torch.float32)
print("Tensor shape", tuple(X.shape), tuple(Y.shape))


Tensor shape (2000, 313, 4) (2000, 313)


In [5]:
try:
    from torch_geometric.nn import GATConv
    class GATRegressor(torch.nn.Module):
        def __init__(self, in_dim=4, hidden_dim=32, heads=2, dropout=0.1):
            super().__init__()
            self.in_proj = torch.nn.Linear(in_dim, hidden_dim)
            self.gat1 = GATConv(hidden_dim, hidden_dim, heads=heads, dropout=dropout)
            self.gat2 = GATConv(hidden_dim*heads, hidden_dim, heads=1, dropout=dropout)
            self.out = torch.nn.Linear(hidden_dim, 1)
            self.drop = torch.nn.Dropout(dropout)

        def forward(self, x, edge_index):
            x = torch.relu(self.in_proj(x))
            x = self.drop(torch.relu(self.gat1(x, edge_index)))
            x = self.drop(torch.relu(self.gat2(x, edge_index)))
            return self.out(x).squeeze(-1)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = GATRegressor().to(device)
    edge_index = edge_index.to(device)

    idx_train = [i for i, s in enumerate(SPLIT) if s == "train"]
    idx_val = [i for i, s in enumerate(SPLIT) if s == "val"]

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    best = (1e9, None, 0)
    patience, patience_ctr = 20, 0
    history = {"train": [], "val": []}

    t0 = time.time()
    for epoch in range(1, 201):
        model.train(); tr_loss = 0.0
        for i in idx_train:
            xb, yb = X[i].to(device), Y[i].to(device)
            opt.zero_grad()
            pred = model(xb, edge_index)
            loss = torch.nn.functional.mse_loss(pred, yb)
            loss.backward()
            opt.step()
            tr_loss += float(loss)
        tr_loss /= max(len(idx_train), 1)

        model.eval(); va_loss = 0.0
        with torch.no_grad():
            for i in idx_val:
                xb, yb = X[i].to(device), Y[i].to(device)
                pred = model(xb, edge_index)
                va_loss += float(torch.nn.functional.mse_loss(pred, yb))
        va_loss /= max(len(idx_val), 1)

        history["train"].append(tr_loss)
        history["val"].append(va_loss)

        if va_loss < best[0]:
            best = (va_loss, {k:v.cpu() for k,v in model.state_dict().items()}, epoch)
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    duration = time.time() - t0
    model.load_state_dict(best[1])

    ckpt = {
        "in_dim": 4,
        "model_state_dict": model.state_dict(),
        "feature_mean": feat_mean.tolist(),
        "feature_std": feat_std.tolist(),
        "target_mean": tgt_mean,
        "target_std": tgt_std,
        "model_version": "gat_v1",
        "best_val_mse": best[0],
        "best_epoch": best[2],
    }
    out_ckpt = PROJECT_ROOT / "artifacts" / "checkpoints" / "gat_v1.pt"
    out_ckpt.parent.mkdir(parents=True, exist_ok=True)
    torch.save(ckpt, out_ckpt)

    print("Best val mse", best[0], "epoch", best[2], "duration_sec", round(duration,2))
except Exception as e:
    print("Training skipped (likely missing torch-geometric):", e)


Training skipped (likely missing torch-geometric): No module named 'torch_geometric'


In [6]:
# Baselines evaluated on test split snapshots
idx_test = [i for i, s in enumerate(SPLIT) if s == "test"]

y_true = np.concatenate([Y[i].numpy() for i in idx_test]) * tgt_std + tgt_mean

# persistence: y_hat = current travel time proxy from speed (approx)
test_rows = df[df.split == "test"].copy()
length_map = mapping.set_index("road_segment_id")["length_m"].to_dict()
current_tt = (test_rows["road_segment_id"].map(length_map).to_numpy()/1000.0) / np.clip(test_rows.current_speed_kmh.to_numpy(), 1e-3, None) * 3600
persist_hat = current_tt
mean_hat = np.full_like(current_tt, fill_value=tgt_mean)


def mae(a,b): return float(np.mean(np.abs(a-b)))
def rmse(a,b): return float(np.sqrt(np.mean((a-b)**2)))

metrics = {
    "persistence": {"mae": mae(test_rows.target_travel_time_sec.to_numpy(), persist_hat), "rmse": rmse(test_rows.target_travel_time_sec.to_numpy(), persist_hat)},
    "train_mean": {"mae": mae(test_rows.target_travel_time_sec.to_numpy(), mean_hat), "rmse": rmse(test_rows.target_travel_time_sec.to_numpy(), mean_hat)},
}

metric_path = PROJECT_ROOT / "artifacts" / "metrics" / "baseline_metrics.json"
metric_path.parent.mkdir(parents=True, exist_ok=True)
metric_path.write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))


{
  "persistence": {
    "mae": 22.550006091671836,
    "rmse": 41.50395017347521
  },
  "train_mean": {
    "mae": 80.90255096862018,
    "rmse": 121.16874772856247
  }
}


Notebook records actual baseline metrics and stores checkpoint when GAT training is possible in the environment.
